# 🔬 τ-Knowledge 소스 준비 (고급 사용자/운영자용)

이 노트북은 **저작 경로 (authoring path)**의 첫 번째 단계입니다.  
학습자 노트북(03, 04)에서 사용하는 **준비된 데이터 번들**을 만들기 위한 소스를 준비합니다.

## 대상
- 운영자 / 고급 사용자 / 데이터 엔지니어
- SDG(합성 데이터 생성)를 직접 수행하려는 사용자

## 처리 단계

1. τ-bench 버전 고정 및 KB(지식 기반) 검사
2. 정책 사실(facts) 추출
3. 도구 스키마 검사
4. 공식 태스크/분할 경계 검토
5. 소스 스냅샷 저장

### 핵심 원칙
- **평가 비공개 정보**(기대 행동, 보상 기준, 숨겨진 사용자 목표)는 학습 데이터에 포함하지 않습니다
- 공식 평가 태스크는 평가용으로 보존하고, 독립적인 학습 시나리오를 생성합니다
- 모든 소스에 대해 버전, SHA, 라이선스를 기록합니다

In [ ]:
"""환경 부트스트랩 — local과 workbench 모두 지원."""

import subprocess, sys, os
from pathlib import Path

# 프로젝트 루트 탐색
_nb_dir = Path.cwd()
_project_root = _nb_dir
for _p in [_nb_dir] + list(_nb_dir.parents):
    if (_p / "pyproject.toml").exists():
        _project_root = _p
        break

# 프로젝트 루트를 환경변수로 설정 (config 모듈이 참조)
os.environ["RHOAI_PROJECT_ROOT"] = str(_project_root)

# 패키지 설치 확인 및 자동 설치
try:
    import rhoai_model_training_lab  # noqa: F401
    print("✅ rhoai_model_training_lab 패키지 확인됨 (extras: sdg)")
except ImportError:
    print("📦 패키지 설치 중... (extras: sdg)")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", ".[sdg]",
         "--extra-index-url", "https://pypi.org/simple/"],
        cwd=str(_project_root),
    )
    print("✅ 설치 완료 (extras: sdg)")


In [ ]:
"""τ-bench 버전 고정 및 KB 검사."""

import os
import json
import subprocess
from pathlib import Path

from rhoai_model_training_lab.config import (
    load_env, load_yaml_config, PROJECT_ROOT,
)

load_env()

prep_config = load_yaml_config("configs/data-preparation.yaml")
tau_config = prep_config["tau_bench"]

tau_version = tau_config.get("version", os.environ.get("TAU_BENCH_VERSION", ""))
tau_sha = tau_config.get("commit_sha", os.environ.get("TAU_BENCH_COMMIT_SHA", ""))
tau_domain = tau_config.get("domain", "banking_knowledge")

def rel(p: Path) -> str:
    """PROJECT_ROOT 기준 상대 경로 (출력용)."""
    try:
        return str(p.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(p)

print("=" * 70)
print("📌 τ-bench 버전 고정")
print("=" * 70)
print(f"  버전: {tau_version}")
print(f"  커밋 SHA: {tau_sha or '(미설정 — 검증 후 고정 필요)'}")
print(f"  도메인: {tau_domain}")
print()

# τ2-bench 데이터 디렉토리 설정 (import 전에 반드시 설정)
tau2_data_dir = PROJECT_ROOT / "vendor" / "tau2-bench" / "data"
if not tau2_data_dir.exists():
    print("📥 τ2-bench 데이터 디렉토리 클론 중...")
    clone_dir = PROJECT_ROOT / "vendor" / "tau2-bench"
    clone_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", tau_version,
         "https://github.com/sierra-research/tau2-bench.git",
         str(clone_dir)],
        check=True,
    )
    print(f"  ✅ 클론 완료: {rel(clone_dir)}")

os.environ["TAU2_DATA_DIR"] = str(tau2_data_dir)
print(f"  TAU2_DATA_DIR = {rel(tau2_data_dir)}")

# τ2-bench 패키지 확인
tau_available = False
try:
    from importlib.metadata import version as pkg_version
    tau_installed = pkg_version("tau2")
    tau_available = True
    print(f"  ✅ τ2-bench 패키지: v{tau_installed}")
except Exception:
    print("  ❌ τ2-bench 미설치. 실행: uv sync --extra sdg")

print("\n📋 버전 참고: v1.0.1에는 banking 채점 수정이 포함되어 있습니다.")


In [ ]:
"""KB 스냅샷 및 정책 사실 추출."""

import sys
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.prepare_tau_sources import (
    inspect_tau_bench, extract_kb_snapshot, extract_policy_facts,
)

source_config = prep_config["sources"]
kb_output = PROJECT_ROOT / source_config["kb_snapshot"]["output_path"]
policy_output = PROJECT_ROOT / source_config["policy_extraction"]["output_path"]

print("=" * 70)
print("📋 Step 1: τ-bench 도메인 검사")
print("=" * 70)

domain_info = inspect_tau_bench(tau_config)
print(f"  KB 발견: {domain_info['kb_found']}")
print(f"  정책 발견: {domain_info['policies_found']}")
print(f"  도구 발견: {domain_info['tools_found']}")

print()
print("=" * 70)
print("📄 Step 2: KB 스냅샷 저장")
print("=" * 70)

kb_count = extract_kb_snapshot(domain_info, kb_output)
print(f"  ✅ KB 문서 {kb_count}개 → {rel(kb_output)}")

print()
print("=" * 70)
print("📋 Step 3: 정책 사실 추출")
print("=" * 70)

policy_count = extract_policy_facts(domain_info, policy_output)
print(f"  ✅ 정책 사실 {policy_count}개 → {rel(policy_output)}")


In [ ]:
"""도구 스키마 추출 및 평가 태스크 예약."""

from scripts.prepare_tau_sources import (
    extract_tool_schemas, reserve_eval_tasks,
)

output_base = PROJECT_ROOT / prep_config["output"]["base_path"]
split_config = prep_config["splits"]

print("=" * 70)
print("🔧 Step 4: 도구 스키마 추출")
print("=" * 70)

tools_output = output_base / "tools" / "schemas.jsonl"
tool_count = extract_tool_schemas(domain_info, tools_output)
print(f"  ✅ 도구 스키마 {tool_count}개 → {rel(tools_output)}")

print()
print("=" * 70)
print("📊 Step 5: 평가 태스크 예약 및 분할 경계 설정")
print("=" * 70)

splits_output = output_base / "splits" / "split_boundaries.json"
split_stats = reserve_eval_tasks(domain_info, split_config, splits_output)
print(f"  평가 예약: {split_stats.get('eval_reserved', 0)}개")
print(f"  학습 후보: {split_stats.get('train_candidates', 0)}개")
print(f"  ✅ 경계 저장 → {rel(splits_output)}")


In [ ]:
"""Review official task/split boundaries."""

print("=" * 70)
print("📊 공식 태스크/분할 경계 검토")
print("=" * 70)

split_config = prep_config["splits"]

print(f"  분할 방법: {split_config['method']}")
print(f"  학습 비율: {split_config['train_ratio']}")
print(f"  검증 비율: {split_config['validation_ratio']}")
print(f"  시드: {split_config['seed']}")
print(f"  오염 검사: {split_config['contamination_check']}")
print(f"  패밀리 격리: {split_config['family_isolation']}")

print("\n📋 분할 원칙:")
print("  1. 공식 평가 태스크는 모두 평가용으로 보존")
print("  2. 공식 학습 분할이 있으면 허용 조건 확인 후 사용")
print("  3. 없으면 KB와 독립 시나리오에서 학습/검증 데이터 생성")
print("  4. 패러프레이즈, 이름/번호 치환, 형제 예제는 같은 분할에 유지")
print("  5. 평가 태스크 문구, 기대 행동, 골든 문서 목록은 절대 SDG에 사용 불가")

print("\n⚠️  핵심 구분:")
print("  • kb_adaptation: KB를 학습하고 새 상황에 적용 → 본 실험")
print("  • 태스크 누출(leakage): 평가 태스크/시나리오를 학습에 사용 → 금지")
print("  • KB 공유는 kb_adaptation에서 의도적이며, 태스크 누출과 구별됨")

In [ ]:
"""소스 스냅샷 저장 및 요약."""

output_base = PROJECT_ROOT / prep_config["output"]["base_path"]
output_base.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("💾 소스 스냅샷 저장")
print("=" * 70)

snapshot_meta = {
    "tau_version": tau_version,
    "tau_commit_sha": tau_sha,
    "domain": tau_domain,
    "extracted": {
        "kb_documents": kb_count if "kb_count" in dir() else 0,
        "policy_facts": policy_count if "policy_count" in dir() else 0,
        "tool_schemas": tool_count if "tool_count" in dir() else 0,
        "eval_tasks_reserved": split_stats.get("eval_reserved", 0) if "split_stats" in dir() else 0,
    },
    "split_config": prep_config["splits"],
    "quality_gates": prep_config.get("quality_gates", {}),
}

meta_path = output_base / "snapshot_metadata.json"
with open(meta_path, "w") as f:
    json.dump(snapshot_meta, f, indent=2, ensure_ascii=False)

print(f"  ✅ 메타데이터: {rel(meta_path)}")
print(f"  출력 경로: {rel(output_base)}")

quality_gates = prep_config.get("quality_gates", {})
print(f"\n📊 품질 게이트:")
print(f"  최소 수용률: {quality_gates.get('min_acceptance_rate', 'N/A')}")
print(f"  필수 유형: {quality_gates.get('required_types', [])}")

print(f"\n다음 단계:")
print(f"  📓 02_generate_synthetic.ipynb — 합성 데이터 생성")
